In [ ]:
# Start for Exercise 8.8 a-c

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error

# Load Carseats data
Carseats = load_data("Carseats")

# (a) Train/test split
np.random.seed(1234)
train, test = train_test_split(Carseats, test_size=0.5, random_state=123)

X_train = train.drop(columns="Sales")
y_train = train["Sales"]
X_test = test.drop(columns="Sales")
y_test = test["Sales"]

# Encode categorical predictors
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)


In [ ]:
# (b) Fit regression tree
tree1 = DecisionTreeRegressor(random_state=123,min_samples_leaf=30)
tree1.fit(X_train, y_train)
# Plot tree
fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(tree1, feature_names=X_train.columns, precision=2, fontsize=10, ax=ax)
plt.show()

In [ ]:
# Predict and compute test MSE
pred1 = tree1.predict(X_test)
testMSE1 = mean_squared_error(y_test, pred1)
print("Test MSE (full tree):", testMSE1)

In [ ]:
# (c) First a large tree
tree2 = DecisionTreeRegressor(min_samples_leaf=2, random_state=0)
tree2.fit(X_train, y_train)
pred2 = tree2.predict(X_test)
fig, ax = plt.subplots(figsize=(12,7))
plot_tree(tree2,feature_names=X_train.columns, precision=3, fontsize=8, 
    ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Pruning
cv_tree = tree2.cost_complexity_pruning_path(X_train,y_train)
kfold = KFold(5,shuffle=True,random_state=10)
grid = GridSearchCV(tree2,{'ccp_alpha': cv_tree.ccp_alphas},
                refit=True,
                cv=kfold,
                scoring="neg_mean_squared_error")
G = grid.fit(X_train,y_train)                
pruned_tree = G.best_estimator_
pred_pruned = pruned_tree.predict(X_test)
fig, ax = plt.subplots(figsize=(12,7))
plot_tree(pruned_tree,feature_names=X_train.columns, precision=3, fontsize=8, 
    ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Optimal alpha
best_alpha = grid.best_params_['ccp_alpha']
print("Best alpha from CV:", best_alpha)

In [ ]:

# Pruned tree with best alpha
pruned_tree.fit(X_train, y_train)
pred3 = pruned_tree.predict(X_test)
testMSE2 = mean_squared_error(y_test, pred2)
testMSE3 = mean_squared_error(y_test, pred3)
print("Test MSE (original):", testMSE2)
print("Test MSE (pruned tree):", testMSE3)